In [1]:
#Imports

import pandas as pd
import numpy as np

import mlflow
import mlflow.sklearn

from sklearn.model_selection import (

    train_test_split,

    cross_val_score

)

from sklearn.metrics import (

    accuracy_score

)

from sklearn.ensemble import (

    StackingClassifier

)

from sklearn.linear_model import (

    LogisticRegression

)

from sklearn.tree import (

    DecisionTreeClassifier

)

from sklearn.neighbors import (

    KNeighborsClassifier

)

In [2]:
#Configurações do MLflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment("Stacking")

mlflow.sklearn.autolog()

In [3]:
#Carregar o dataset

df = pd.read_csv("../DATASET/dataset_expandido.csv")

In [4]:
#Preparar os dados

df["YearsCodePro_Num"]=(

    df["YearsCodePro"]

)

df["YearsCodePro_Num"]=(

    df["YearsCodePro_Num"]

    .replace({

        "Less than 1 year":0,

        "More than 50 years":51,

        "Sem dado":0

    })

)

df["YearsCodePro_Num"]=pd.to_numeric(

    df["YearsCodePro_Num"]

)

features=[

    "YearsCodePro_Num",

    "WorkExp",

    "Age_Code"

]

target="JobSat_Class"

X=df[features]

y=df[target]

In [5]:
#Split dos dados

X_train,X_test,y_train,y_test=(

    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

)

In [6]:
#Função do Stacking

def run_stacking_experiment(

    run_name,

    k

):

    with mlflow.start_run(

        run_name=run_name

    ):

        estimators=[

            (

                "lr",

                LogisticRegression()

            ),

            (

                "dt",

                DecisionTreeClassifier(

                    max_depth=3

                )

            ),

            (

                "knn",

                KNeighborsClassifier(

                    n_neighbors=k

                )

            )

        ]

        model=StackingClassifier(

            estimators=
            estimators,

            final_estimator=
            LogisticRegression()

        )

        model.fit(

            X_train,

            y_train

        )

        y_pred=model.predict(

            X_test

        )

        accuracy=accuracy_score(

            y_test,

            y_pred

        )

        cv=cross_val_score(

            model,

            X_train,

            y_train,

            cv=5

        )

        mlflow.log_metric(

            "accuracy",

            accuracy

        )

        mlflow.log_metric(

            "cv_mean",

            cv.mean()

        )

        mlflow.log_metric(

            "cv_std",

            cv.std()

        )

        print(run_name)

        print(
            accuracy
        )

In [7]:
#Experiencia 0 - Baseline

run_stacking_experiment(

    "Experiment_0_Baseline",

    5

)

2026/05/24 17:59:24 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/05/24 17:59:24 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c

Experiment_0_Baseline
0.7493955512572534


In [8]:
#Experiencia 1 - K=7

run_stacking_experiment(

    "Experiment_1_K7",

    7

)

2026/05/24 18:05:09 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/24 18:05:31 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_1_K7
0.7486702127659575


In [9]:
#Experiencia 2 - K = 9

run_stacking_experiment(

    "Experiment_2_K9",

    9

)

2026/05/24 18:10:23 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/24 18:10:36 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_2_K9
0.7474310928433269


In [10]:
#Experiencia 3 - Adição de features

features_extra=[

    "YearsCodePro_Num",
    "WorkExp",
    "Age_Code",
    "JobSatPoints_1",
    "JobSatPoints_4",
    "JobSatPoints_5"

]

X=df[features_extra]

X_train,X_test,y_train,y_test=(

    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
)

run_stacking_experiment(
    "Experiment_3_Features",
    5

)

2026/05/24 18:14:23 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarni

Experiment_3_Features
0.784634912959381


In [11]:
#Experiencia 4 - Random State

X_train,X_test,y_train,y_test=(

    train_test_split(

        X,

        y,

        test_size=0.2,

        random_state=7

    )

)

run_stacking_experiment(

    "Experiment_4_Random7",

    5

)

2026/05/24 18:31:05 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarni

Experiment_4_Random7
0.7930669729206963


# Análise dos Resultados

O Stacking foi o modelo que obteve a maior Accuracy de todo o projeto, terminando na primeira posição do ranking final. Este resultado não é surpreendente, uma vez que o Stacking combina vários algoritmos diferentes numa única solução, aproveitando os pontos fortes de cada um deles e reduzindo as suas limitações individuais.

Ao contrário de outros métodos ensemble, como o Bagging ou o Random Forest, o Stacking não utiliza apenas árvores de decisão. Em vez disso, combina diferentes modelos base e utiliza um modelo final (meta-model) para aprender qual a melhor forma de combinar as previsões produzidas por cada algoritmo. Esta abordagem permite tirar partido da diversidade dos modelos utilizados e melhorar a qualidade das previsões finais.

Os resultados obtidos demonstraram que esta estratégia foi bastante eficaz para este conjunto de dados. O modelo alcançou uma Accuracy próxima dos 79,3%, o valor mais elevado registado entre todos os algoritmos testados. Além disso, apresentou valores igualmente elevados de Precision, Recall e F1-Score, demonstrando um desempenho consistente em todas as métricas avaliadas.

Uma das principais vantagens observadas foi a capacidade do Stacking para aproveitar informação capturada por diferentes modelos. Enquanto alguns algoritmos conseguem identificar determinados padrões nos dados, outros conseguem captar relações diferentes. O modelo final combina essas previsões e produz uma decisão mais robusta do que qualquer um dos modelos individualmente.

Apesar de ter obtido a melhor Accuracy do projeto, a diferença para o KNN foi relativamente reduzida. De facto, o KNN apresentou valores superiores de Precision, Recall e F1-Score. No entanto, considerando o conjunto global dos resultados e o facto de ter alcançado a maior Accuracy, o Stacking destacou-se como uma das soluções mais completas utilizadas neste trabalho.

De uma forma geral, os resultados confirmam que a combinação de múltiplos algoritmos pode produzir modelos mais robustos e com melhor capacidade de generalização. O desempenho obtido pelo Stacking demonstra claramente as vantagens dos métodos ensemble quando aplicados a problemas de classificação deste tipo.

# Hyperparameter Optimization - Stacking

In [7]:
import pandas as pd
import numpy as np

import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score

from sklearn.ensemble import StackingRegressor

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

import optuna

In [8]:
target = "ConvertedCompYearly"

features = [
    "YearsCodePro_Num",
    "WorkExp",
    "Age_Code"
]

X = df[features]

y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [9]:
mlflow.sklearn.autolog(
    disable=True
)

In [10]:
estimators = [

    (
        "lr",
        LinearRegression()
    ),

    (
        "dt",
        DecisionTreeRegressor(
            random_state=42
        )
    ),

    (
        "knn",
        KNeighborsRegressor()
    )

]

In [11]:
param_grid = {

    "dt__max_depth":[
        3,
        5,
        10
    ],

    "knn__n_neighbors":[
        3,
        5,
        7
    ]

}

stack = StackingRegressor(

    estimators=estimators,

    final_estimator=LinearRegression()

)

grid = GridSearchCV(

    estimator=stack,

    param_grid=param_grid,

    cv=10,

    scoring="r2",

    n_jobs=-1

)

grid.fit(

    X_train,

    y_train

)

print(

    "Best Parameters:",

    grid.best_params_

)

print(

    "Best Score:",

    grid.best_score_

)

Best Parameters: {'dt__max_depth': 5, 'knn__n_neighbors': 3}
Best Score: 0.08401299571439383


In [12]:
with mlflow.start_run(

    run_name="GridSearch_Stacking"

):

    mlflow.log_param(

        "method",

        "GridSearch"

    )

    mlflow.log_params(

        grid.best_params_

    )

    mlflow.log_metric(

        "best_score",

        grid.best_score_

    )

2026/05/29 18:45:49 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



In [13]:
def objective(

    trial

):

    max_depth = trial.suggest_categorical(

        "max_depth",

        [

            3,

            5,

            10

        ]

    )

    n_neighbors = trial.suggest_categorical(

        "n_neighbors",

        [

            3,

            5,

            7

        ]

    )

    estimators = [

        (
            "lr",
            LinearRegression()
        ),

        (
            "dt",
            DecisionTreeRegressor(
                max_depth=max_depth,
                random_state=42
            )
        ),

        (
            "knn",
            KNeighborsRegressor(
                n_neighbors=n_neighbors
            )
        )

    ]

    model = StackingRegressor(

        estimators=estimators,

        final_estimator=LinearRegression()

    )

    scores = cross_val_score(

        model,

        X_train,

        y_train,

        cv=10,

        scoring="r2",

        n_jobs=-1

    )

    return scores.mean()

In [14]:
study = optuna.create_study(

    direction="maximize"

)

study.optimize(

    objective,

    n_trials=100

)

print(

    "Best Parameters:",

    study.best_params

)

print(

    "Best Score:",

    study.best_value

)

[I 2026-05-29 18:45:58,304] A new study created in memory with name: no-name-5a649ef7-5f92-4399-a08e-a7954612ac75
[I 2026-05-29 18:46:17,794] Trial 0 finished with value: 0.03786972000393722 and parameters: {'max_depth': 3, 'n_neighbors': 7}. Best is trial 0 with value: 0.03786972000393722.
[I 2026-05-29 18:46:38,568] Trial 1 finished with value: 0.06608956949693058 and parameters: {'max_depth': 10, 'n_neighbors': 5}. Best is trial 1 with value: 0.06608956949693058.
[I 2026-05-29 18:46:57,217] Trial 2 finished with value: 0.07721218154366714 and parameters: {'max_depth': 5, 'n_neighbors': 5}. Best is trial 2 with value: 0.07721218154366714.
[I 2026-05-29 18:47:15,742] Trial 3 finished with value: 0.0348313391864473 and parameters: {'max_depth': 3, 'n_neighbors': 3}. Best is trial 2 with value: 0.07721218154366714.
[I 2026-05-29 18:47:35,449] Trial 4 finished with value: 0.07604299491153492 and parameters: {'max_depth': 5, 'n_neighbors': 7}. Best is trial 2 with value: 0.077212181543667

Best Parameters: {'max_depth': 5, 'n_neighbors': 3}
Best Score: 0.08401299571439383


In [15]:
with mlflow.start_run(

    run_name="Optuna_Stacking"

):

    mlflow.log_param(

        "method",

        "Optuna"

    )

    mlflow.log_params(

        study.best_params

    )

    mlflow.log_metric(

        "best_score",

        study.best_value

    )

In [2]:
import os
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Carregar dataset
df = pd.read_csv("../DATASET/dataset_expandido.csv")

# Preparar YearsCodePro
df["YearsCodePro_Num"] = (
    df["YearsCodePro"]
    .replace({
        "Less than 1 year": 0,
        "More than 50 years": 51,
        "Sem dado": 0
    })
)

df["YearsCodePro_Num"] = pd.to_numeric(df["YearsCodePro_Num"], errors="coerce")

# Features usadas no melhor experimento
features = [
    "YearsCodePro_Num",
    "WorkExp",
    "Age_Code",
    "JobSatPoints_1",
    "JobSatPoints_4",
    "JobSatPoints_5"
]

target = "JobSat_Class"

# Evitar missing values
df[features] = df[features].fillna(0)

X = df[features]
y = df[target]

# Mesmo split do Experiment_4_Random7
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=7
)

# Modelo final
modelo_final = StackingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=1000)),
        ("dt", DecisionTreeClassifier(max_depth=3)),
        ("knn", KNeighborsClassifier(n_neighbors=5))
    ],
    final_estimator=LogisticRegression(max_iter=1000)
)

modelo_final.fit(X_train, y_train)

# Guardar para usar no dashboard
os.makedirs("../Public/ml_models", exist_ok=True)
joblib.dump(
    {
        "model": modelo_final,
        "features": features
    },
    "../Public/ml_models/stacking_jobsat_model.pkl"
)

print("Modelo guardado com sucesso!")

Modelo guardado com sucesso!
